In [1]:
import os

def setting_up_proxy(proxy=None, proxy_type='http', verbose=True):
    supported_proxy_types = ['http', 'https', 'socks4', 'socks5', 'all']
    assert proxy_type in supported_proxy_types, f"proxy type {repr(proxy_type)} not supported, only support {supported_proxy_types}"
    if proxy is None:
        proxy = os.environ.get(f'{proxy_type}_proxy')
    if proxy is None:
        return
    if verbose:
        print(f'setting up proxy {repr(proxy)} for {repr(proxy_type)}')
    os.environ[f'{proxy_type}_proxy'] = proxy


default_proxy_config = {
    'http': 'http://127.0.0.1:7890',
    'https': 'http://127.0.0.1:7890',
    'all': 'socks5://127.0.0.1:7890',
}


# default_proxy_config = {
#     'http': 'http://10.176.52.116:7890',
#     'https': 'http://10.176.52.116:7890',
#     'all': 'socks5://10.176.52.116:7890',
# }


def setting_up_proxy_from_config(proxy_config=default_proxy_config, verbose=True):
    for proxy_type, proxy_url in proxy_config.items():
        setting_up_proxy(proxy=proxy_url, proxy_type=proxy_type, verbose=verbose)
    print()


# setting_up_proxy_from_config()

In [2]:
import requests
resp = requests.get('https://papers.cool/venue/ICLR.2024?group=Poster&show=2000', proxies=default_proxy_config)
print(resp)
print(resp.headers['content-type'])
with open('iclr2024_2000.html', 'wb') as f:
    f.write(resp.content)

<Response [200]>
text/html; charset=UTF-8


In [3]:
from RFC.item.item import ItemType
from RFC.utils.parse import (
    get_html_soup,
    parse,
)


def parse_kimi_links(resp):
    parse_config = {
        'kimi_links': {
            ('attr', 'a', 'class', 'title-kimi', None): {}
        },
        'pdf_links': {
            ('attr', 'a', 'class', 'title-pdf', None): {}
        },
        'title': {
            ('attr', 'a', 'class', 'title-link', None): {}
        }
    }
    
    # kimi-links
    kimi_link_template = 'https://papers.cool/venue/kimi?paper={}'
    kimi_links_soup = parse(get_html_soup(resp.content), parse_config['kimi_links'])
    kimi_links_id = [soup['id'] for soup in kimi_links_soup]
    kimi_links = []
    for kimi_link_id in kimi_links_id:
        assert kimi_link_id.startswith('kimi-')
        kimi_links.append(kimi_link_template.format(kimi_link_id[len('kimi-'):]))
    
    # pdf-links
    pdf_links_soup = parse(get_html_soup(resp.content), parse_config['pdf_links'])
    raw_pdf_links = [soup['onclick'].split(', ')[1] for soup in pdf_links_soup]
    pdf_links = []
    for raw_pdf_link in raw_pdf_links:
        if raw_pdf_link.startswith('\''):
            raw_pdf_link = raw_pdf_link[1:]
        if raw_pdf_link.endswith('\''):
            raw_pdf_link = raw_pdf_link[:-1]
        if raw_pdf_link.startswith('/pdf?url='):
            raw_pdf_link = raw_pdf_link[len('/pdf?url='):]
        assert raw_pdf_link.startswith('http')
        pdf_links.append(raw_pdf_link)
        
    # title
    title_soup = parse(get_html_soup(resp.content), parse_config['title'])
    titles = [soup.text for soup in title_soup]
    
    assert len(kimi_links) == len(pdf_links) == len(titles)
    return kimi_links, pdf_links, titles


kimi_links, pdf_links, titles = parse_kimi_links(resp)
print(len(kimi_links), kimi_links[:10])
print(len(pdf_links), pdf_links[:10])
print(len(titles), titles[:10])

2024-04-07 02:57:18,892 <MainProcess:952992, MainThread:140564296820544> (<frozen importlib._bootstrap>:_call_with_frames_removed:241 -> arg.py:<module>:39) from logger `RFC.args.arg`
[INFO] importing module RFC.args.arg

2024-04-07 02:57:19,868 <MainProcess:952992, MainThread:140564296820544> (<frozen importlib._bootstrap>:_call_with_frames_removed:241 -> arg.py:<module>:331) from logger `RFC.args.arg`
[INFO] module RFC.args.arg imported

2024-04-07 02:57:19,974 <MainProcess:952992, MainThread:140564296820544> (<frozen importlib._bootstrap>:_call_with_frames_removed:241 -> arg_group.py:<module>:38) from logger `RFC.args.arg_group`
[INFO] importing module RFC.args.arg_group

2024-04-07 02:57:20,017 <MainProcess:952992, MainThread:140564296820544> (<frozen importlib._bootstrap>:_call_with_frames_removed:241 -> arg_group.py:<module>:302) from logger `RFC.args.arg_group`
[INFO] module RFC.args.arg_group imported

2024-04-07 02:57:20,056 <MainProcess:952992, MainThread:140564296820544> (<f

KeyboardInterrupt: 

In [ ]:
import aiohttp
from tqdm import tqdm

for i in tqdm(range(100000)):
    async with aiohttp.request('get', "https://papers.cool", proxy='http://10.176.52.116:7890') as resp:
        with open('cookies.txt', 'a') as f:
            f.write(str({'client_id': dict(resp.cookies)['client_id'].value})+'\n')

  0%|          | 0/100000 [00:00<?, ?it/s]

  0%|          | 109/100000 [01:01<15:40:26,  1.77it/s]


CancelledError: 

In [20]:
import requests
from tqdm import tqdm

# for i in tqdm(range(100000)):
#     resp = requests.get('https://papers.cool', proxies=default_proxy_config)
#     # print(resp)
#     with open('cookies.txt', 'a') as f:
#         f.write(dict(resp.cookies)['client_id']+'\n')


# resp = requests.get('https://papers.cool', proxies=default_proxy_config)
# resp.cookies


# paper="!/RCx5l/IXbQv+OWMzlUW2A==?gAWVGgAAAAAAAACMBXBhcGVylIwMUDE3LTEwMjRAQUNMlIaULg=="

# cookies = dict(
#     client_id="!hsIR1RZIl6Z6onNFYP1/xA==?gAWVKQAAAAAAAACMCWNsaWVudF9pZJSMFzQ5MjMtMTcxMjY0ODQ5NS43NTI0OTI0lIaULg=="
# )


# resp = requests.request('post', 'https://papers.cool/venue/star?key=kimi&paper=P13-1097@ACL', cookies=cookies, proxies=default_proxy_config)
# dict(resp.cookies)

cookies = {'client_id': '"!PtlkCk4+xQRayHiSNGVtBQ==?gAWVKQAAAAAAAACMCWNsaWVudF9pZJSMFzUwODgtMTcxMjY0ODkzMi4zMTcyMTMzlIaULg=="'}
resp = requests.request('post', 'https://papers.cool/venue/star?key=kimi&paper=P13-1099@ACL', cookies=cookies, proxies=default_proxy_config)
dict(resp.cookies)
# resp = requests.request('get', 'https://papers.cool/venue/NDSS.2024', cookies=cookies, proxies=default_proxy_config)
# dict(resp.cookies)

{'paper': '"!foJFa2N51mNjqIJiN+U72g==?gAWVGgAAAAAAAACMBXBhcGVylIwMUDEzLTEwOTlAQUNMlIaULg=="'}

In [7]:
def get_full_date_tags(st_year=2024, st_month=1, st_day=1):
    from datetime import date, timedelta

    def daterange(start_date, end_date):
        for n in range(int((end_date - start_date).days)):
            yield start_date + timedelta(n)

    start_date = date(st_year, st_month, st_day)
    end_date = date.today() + timedelta(1)
    full_dates = []
    for single_date in daterange(start_date, end_date):
        full_dates.append(single_date.strftime("%Y-%m-%d"))
    return full_dates


get_full_date_tags()

['2024-01-01',
 '2024-01-02',
 '2024-01-03',
 '2024-01-04',
 '2024-01-05',
 '2024-01-06',
 '2024-01-07',
 '2024-01-08',
 '2024-01-09',
 '2024-01-10',
 '2024-01-11',
 '2024-01-12',
 '2024-01-13',
 '2024-01-14',
 '2024-01-15',
 '2024-01-16',
 '2024-01-17',
 '2024-01-18',
 '2024-01-19',
 '2024-01-20',
 '2024-01-21',
 '2024-01-22',
 '2024-01-23',
 '2024-01-24',
 '2024-01-25',
 '2024-01-26',
 '2024-01-27',
 '2024-01-28',
 '2024-01-29',
 '2024-01-30',
 '2024-01-31',
 '2024-02-01',
 '2024-02-02',
 '2024-02-03',
 '2024-02-04',
 '2024-02-05',
 '2024-02-06',
 '2024-02-07',
 '2024-02-08',
 '2024-02-09',
 '2024-02-10',
 '2024-02-11',
 '2024-02-12',
 '2024-02-13',
 '2024-02-14',
 '2024-02-15',
 '2024-02-16',
 '2024-02-17',
 '2024-02-18',
 '2024-02-19',
 '2024-02-20',
 '2024-02-21',
 '2024-02-22',
 '2024-02-23',
 '2024-02-24',
 '2024-02-25',
 '2024-02-26',
 '2024-02-27',
 '2024-02-28',
 '2024-02-29',
 '2024-03-01',
 '2024-03-02',
 '2024-03-03',
 '2024-03-04',
 '2024-03-05',
 '2024-03-06',
 '2024-03-

In [49]:
import time
import requests


headers = {
    'Accept': '*/*',
    'Accept-Encoding': 'gzip, deflate, br, zstd',
    'Accept-Language': 'en-US,en;q=0.9',
    'Cache-Control': 'no-cache',
    'Content-Length': '0',
    'Origin': 'https://papers.cool',
    'Pragma': 'no-cache',
    # 'Referer': 'https://papers.cool/arxiv/cs.AI?show=100',
    'Referer': 'https://papers.cool/venue/ACL.2016?show=50',
    'Sec-Ch-Ua': '"Google Chrome";v="123", "Not:A-Brand";v="8", "Chromium";v="123"',
    'Sec-Ch-Ua-Mobile': '?0',
    'Sec-Ch-Ua-Platform': '"Windows"',
    'Sec-Fetch-Dest': 'empty',
    'Sec-Fetch-Mode': 'cors',
    'Sec-Fetch-Site': 'same-origin',
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/123.0.0.0 Safari/537.36'
}
# default_proxy_config = {
#     'http': 'http://127.0.0.1:10002',
#     'https': 'http://127.0.0.1:10002',
#     'all': 'socks5://127.0.0.1:10002',
# }
default_proxy_config = {
    'http': 'http://127.0.0.1:7890',
    'https': 'http://127.0.0.1:7890',
    'all': 'socks5://127.0.0.1:7890',
}


def lazy_request_cookies_online(*args, **kwargs):
    for i in range(3):
        try:
            resp = requests.request('post', "https://papers.cool/venue/star?key=kimi&paper=P16-1019@ACL", proxies=default_proxy_config)
            print(resp)
            return dict(resp.cookies)
        except Exception as e:
            raise e
            time.sleep(0.5)
            continue
    return {}


lazy_request_cookies_online()

<Response [200]>


{'client_id': '"!aRleIr7Yt2F0DWKN71cWrQ==?gAWVKAAAAAAAAACMCWNsaWVudF9pZJSMFjk1NC0xNzEyNDMxMjA1LjY1Njg3NTiUhpQu"'}

In [22]:
import requests
import aiohttp

headers = {
    'Accept': '*/*',
    'Accept-Encoding': 'gzip, deflate, br, zstd',
    'Accept-Language': 'en-US,en;q=0.9,zh-CN;q=0.8,zh;q=0.7',
    'Cache-Control': 'no-cache',
    'Content-Length': '0',
    'Origin': 'https://papers.cool',
    'Pragma': 'no-cache',
    # 'Referer': 'https://papers.cool/arxiv/cs.AI?show=100',
    'Referer': 'https://papers.cool/venue/ACL.2012?show=100',
    'Sec-Ch-Ua': '"Google Chrome";v="123", "Not:A-Brand";v="8", "Chromium";v="123"',
    'Sec-Ch-Ua-Mobile': '?0',
    'Sec-Ch-Ua-Platform': '"Windows"',
    'Sec-Fetch-Dest': 'empty',
    'Sec-Fetch-Mode': 'cors',
    'Sec-Fetch-Site': 'same-origin',
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/123.0.0.0 Safari/537.36'
}

cookies = {
    'client_id': '"!PtlkCk4+xQRayHiSNGVtBQ==?gAWVKQAAAAAAAACMCWNsaWVudF9pZJSMFzUwODgtMTcxMjY0ODkzMi4zMTcyMTMzlIaULg=="',
    'paper': '"!foJFa2N51mNjqIJiN+U72g==?gAWVGgAAAAAAAACMBXBhcGVylIwMUDEzLTEwOTlAQUNMlIaULg=="',
}

# resp = requests.post('https://papers.cool/venue/star?key=kimi&paper=P14-1019@ACL', cookies=cookies, headers=headers)
# print(resp)
# print(resp.content)
# print('next!')

# venue_ICLR.2023_=-EHqoysUYLx@OpenReview
resp = requests.post('https://papers.cool/venue/kimi?paper=P13-1099@ACL', proxies={'http': 'http://10.176.52.116:10000'}, cookies=cookies)
print(resp)
print(resp.headers['content-type'])
with open('kimi_paper.html', 'wb') as f:
    f.write(resp.content)

# async with aiohttp.request('post', "https://papers.cool/venue/kimi?paper=P13-1033@ACL", proxy='http://10.176.52.116:7890', cookies=cookies) as resp:
#     print(resp)
#     print(resp.headers['content-type'])
#     with open('kimi_paper.html', 'wb') as f:
#         f.write(await resp.read())

    
# for chunk in response.iter_content(chunk_size=1024):
#     print('hit!')
#     print(chunk)

<Response [200]>
text/html; charset=utf-8


In [ ]:
resp = requests.get('https://papers.cool/venue/progress?paper=P13-1022@ACL')
print(resp)
print(resp.content)

<Response [200]>
b'-2066'
